# Trick or Data: Statistical Analysis

## Objective

This notebook investigates the statistical significance and magnitude of relationships identified during the exploratory data analysis.

The analysis focuses on comparing candy popularity across different attributes and evaluating whether the observed differences are supported by statistical evidence.

## Research Questions

1. Is the difference in popularity between chocolate and non-chocolate candies statistically significant?
2. Is the difference between fruity and non-fruity candies statistically significant?
3. How large are the observed differences between these groups?
4. Which candy attributes show the strongest associations with popularity?

## Statistical Methods

The analysis explores:

- Null and alternative hypotheses
- Independent samples t-tests
- Mann–Whitney U tests
- p-values and statistical significance
- Confidence intervals
- Effect sizes

## Dataset

- **Source:** FiveThirtyEight Halloween Candy Power Ranking
- **Input:** Cleaned candy dataset
- **Observations:** 85 candy products
- **Target variable:** `winpercent`
- **Tools:** Python, Pandas, NumPy, SciPy, and Matplotlib

## 1. Data Loading

This section loads the cleaned dataset generated during the data preparation process.

The processed dataset serves as the input for the statistical analyses performed in this notebook.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/Leticia-Sousa-DS/trick-or-data/refs/heads/main/data/processed/candy_data_cleaned.csv"

df = pd.read_csv(url)

In [11]:
print(f"Dataset shape: {df.shape}")
display(df.head())

Dataset shape: (85, 13)


,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent
0,100 Grand,1,0,1,0,0,1,0,1,0,0.732,0.860,66.971725
1,3 Musketeers,1,0,0,0,1,0,0,1,0,0.604,0.511,67.602936
2,One dime,0,0,0,0,0,0,0,0,0,0.011,0.116,32.261086
3,One quarter,0,0,0,0,0,0,0,0,0,0.011,0.511,46.116505
4,Air Heads,0,1,0,0,0,0,0,0,0,0.906,0.511,52.341465


## 2. Hypothesis Formulation

This section defines the null and alternative hypotheses for the statistical comparisons.

### Research Question 1

Is there a statistically significant difference in popularity between chocolate and non-chocolate candies?

- **Null hypothesis (H0):** There is no difference in mean win percentage between chocolate and non-chocolate candies.
- **Alternative hypothesis (H1):** There is a difference in mean win percentage between chocolate and non-chocolate candies.

The test will use a two-sided alternative hypothesis, since the objective is to determine whether a difference exists, regardless of its direction.

## 3. Descriptive Statistics

This section examines the descriptive statistics of chocolate and non-chocolate candies to understand differences in their popularity distributions before conducting hypothesis tests.

The analysis includes sample size, mean, standard deviation, median, minimum, and maximum win percentages.

In [12]:
df.groupby('chocolate')['winpercent'].agg(
    ['count', 'mean', 'std', 'median', 'min', 'max']
)

,count,mean,std,median,min,max
chocolate,,,,,,
0,48,42.142257,10.220959,41.646933,22.445341,67.037628
1,37,60.921529,12.811124,60.800701,34.722000,84.180290


## 4. Statistical Test Selection

This section defines the statistical methods used to evaluate the hypotheses formulated in the previous section.

The Welch's t-test is selected as the primary method to compare mean win percentages between chocolate and non-chocolate candies.

This test compares two independent groups without assuming equal population variances.

The Mann–Whitney U test is also used as a complementary non-parametric method to compare the distributions of the two groups.

Both tests use a two-sided alternative hypothesis, as the objective is to investigate whether a difference exists, regardless of its direction.

In [13]:
# Separate candies into chocolate and non-chocolate groups
chocolate_group = df[df['chocolate'] == 1]['winpercent']
non_chocolate_group = df[df['chocolate'] == 0]['winpercent']

# Welch's t-test
welch_result = stats.ttest_ind(
    chocolate_group,
    non_chocolate_group,
    equal_var=False,
    alternative='two-sided'
)

# Mann–Whitney U test
mannwhitney_result = stats.mannwhitneyu(
    chocolate_group,
    non_chocolate_group,
    alternative='two-sided'
)



## 5. Statistical Test Results

Both statistical tests indicate a statistically significant difference between chocolate and non-chocolate candies.

The Welch's t-test evaluates the difference in mean popularity, while the Mann–Whitney U test provides a complementary comparison based on the ranks of the observations.

The results provide evidence against the null hypothesis at the 5% significance level.

However, statistical significance does not necessarily indicate the practical magnitude of a difference, nor does it establish a causal relationship between chocolate content and candy popularity.

In [14]:
print("Welch's t-test")
print(f"t-statistic: {welch_result.statistic:.4f}")
print(f"p-value: {welch_result.pvalue:.6f}")

print("\nMann–Whitney U test")
print(f"U-statistic: {mannwhitney_result.statistic:.4f}")
print(f"p-value: {mannwhitney_result.pvalue:.6f}")

print("\nP-values in scientific notation")
print(f"Welch's t-test: {welch_result.pvalue:.10e}")
print(f"Mann–Whitney U test: {mannwhitney_result.pvalue:.10e}")

Welch's t-test
t-statistic: 7.3031
p-value: 0.000000

Mann–Whitney U test
U-statistic: 1542.0000
p-value: 0.000000

P-values in scientific notation
Welch's t-test: 4.1635072177e-10
Mann–Whitney U test: 6.9349267098e-09


## 6. Confidence Interval

This section estimates the difference in mean popularity between chocolate and non-chocolate candies using a 95% confidence interval.

The interval is calculated using the Welch-Satterthwaite method, which accounts for potentially unequal variances and sample sizes between the two groups.

The confidence interval provides a range of plausible values for the difference in population means under the assumptions of the statistical model.

In [15]:
# Calculate the difference between group means
mean_difference = chocolate_group.mean() - non_chocolate_group.mean()

# Calculate the standard error
standard_error = np.sqrt(
    (chocolate_group.var(ddof=1) / len(chocolate_group)) +
    (non_chocolate_group.var(ddof=1) / len(non_chocolate_group))
)

# Calculate Welch-Satterthwaite degrees of freedom
numerator = (
    (chocolate_group.var(ddof=1) / len(chocolate_group)) +
    (non_chocolate_group.var(ddof=1) / len(non_chocolate_group))
) ** 2

denominator = (
    (chocolate_group.var(ddof=1) / len(chocolate_group)) ** 2 /
    (len(chocolate_group) - 1)
) + (
    (non_chocolate_group.var(ddof=1) / len(non_chocolate_group)) ** 2 /
    (len(non_chocolate_group) - 1)
)

degrees_of_freedom = numerator / denominator

# Calculate the 95% confidence interval
confidence_level = 0.95
alpha = 1 - confidence_level

critical_value = stats.t.ppf(
    1 - alpha / 2,
    degrees_of_freedom
)

margin_of_error = critical_value * standard_error

lower_bound = mean_difference - margin_of_error
upper_bound = mean_difference + margin_of_error

print(f"Mean difference: {mean_difference:.4f}")
print(f"Standard error: {standard_error:.4f}")
print(f"Degrees of freedom: {degrees_of_freedom:.2f}")
print(f"95% confidence interval: [{lower_bound:.4f}, {upper_bound:.4f}]")

Mean difference: 18.7793
Standard error: 2.5714
Degrees of freedom: 67.54
95% confidence interval: [13.6474, 23.9111]


### Confidence Interval Interpretation

The estimated mean difference in popularity between chocolate and non-chocolate candies is approximately 18.78 percentage points.

The 95% confidence interval ranges from 13.65 to 23.91 percentage points. Since the interval does not include zero, it is consistent with the statistically significant result obtained from Welch's t-test.

This suggests that chocolate candies have a higher mean win percentage in the analyzed dataset.

The confidence interval quantifies uncertainty around the estimated difference, but it does not establish a causal relationship between chocolate content and popularity.

## 7. Effect Size

This section measures the magnitude of the difference in popularity between chocolate and non-chocolate candies.

Hedges' g is used to estimate the standardized mean difference between the two groups, applying a correction for small-sample bias.

Unlike the p-value, which evaluates statistical evidence, the effect size helps assess the magnitude of the observed difference.

The pooled standard deviation is used as the standardizer for this comparison.

In [16]:
# Calculate the pooled standard deviation
pooled_std = np.sqrt(
    (
        (len(chocolate_group) - 1) * chocolate_group.var(ddof=1) +
        (len(non_chocolate_group) - 1) * non_chocolate_group.var(ddof=1)
    ) /
    (len(chocolate_group) + len(non_chocolate_group) - 2)
)

# Calculate Cohen's d
cohens_d = mean_difference / pooled_std

# Apply the small-sample correction
degrees_of_freedom_pooled = (
    len(chocolate_group) + len(non_chocolate_group) - 2
)

correction_factor = 1 - (3 / (4 * degrees_of_freedom_pooled - 1))

# Calculate Hedges' g
hedges_g = cohens_d * correction_factor

print(f"Pooled standard deviation: {pooled_std:.4f}")
print(f"Cohen's d: {cohens_d:.4f}")
print(f"Correction factor: {correction_factor:.4f}")
print(f"Hedges' g: {hedges_g:.4f}")

Pooled standard deviation: 11.4168
Cohen's d: 1.6449
Correction factor: 0.9909
Hedges' g: 1.6300


### Effect Size Interpretation

The calculated Hedges' g is approximately 1.63, indicating a large standardized difference in popularity between chocolate and non-chocolate candies.

This result suggests that the difference in group means is substantial relative to the pooled standard deviation.

The small-sample correction has only a minor impact on the estimate, reducing Cohen's d from 1.6449 to 1.6300.

These conventional effect size thresholds should be interpreted as guidelines rather than universal rules.

Since the primary hypothesis test uses Welch's t-test, which does not assume equal population variances, Hedges' g should be interpreted as a descriptive standardized comparison based on the pooled standard deviation.

The effect size does not establish causation or account for other candy attributes.

## 8. Additional Attribute Analysis

This section investigates the relationship between other candy attributes and popularity (`winpercent`).

The analysis examines differences in mean popularity between candies with and without selected attributes.

The objective is to identify additional patterns and compare their magnitude with the results observed for chocolate.

These comparisons are exploratory and do not establish causal relationships. Since multiple attributes may coexist in the same candy, the observed differences should not be interpreted as independent effects.

### 8.1 Comparing Candy Attributes

For each binary attribute, the mean win percentage is compared between candies that have the attribute and those that do not.

Welch's t-test is used to evaluate the differences in group means.

Since multiple statistical tests are performed, the Holm correction is applied to adjust the p-values and control the family-wise error rate.

The results are exploratory and should be interpreted with consideration for overlapping candy attributes and the relatively small dataset.

In [17]:
# Define the attributes to investigate, excluding chocolate
attributes = [
    'fruity',
    'caramel',
    'peanutyalmondy',
    'nougat',
    'crispedricewafer',
    'hard',
    'bar',
    'pluribus'
]

results = []

# Compare mean popularity for each attribute
for attribute in attributes:
    group_with = df[df[attribute] == 1]['winpercent']
    group_without = df[df[attribute] == 0]['winpercent']

    test_result = stats.ttest_ind(
        group_with,
        group_without,
        equal_var=False,
        alternative='two-sided'
    )

    results.append({
        'attribute': attribute,
        'count_with': len(group_with),
        'count_without': len(group_without),
        'mean_with': group_with.mean(),
        'mean_without': group_without.mean(),
        'mean_difference': group_with.mean() - group_without.mean(),
        'p_value': test_result.pvalue
    })

# Create results DataFrame
attribute_results = pd.DataFrame(results)

# Apply Holm correction
sorted_indices = np.argsort(attribute_results['p_value'].values)
sorted_pvalues = attribute_results['p_value'].values[sorted_indices]

adjusted_pvalues = np.maximum.accumulate(
    (len(sorted_pvalues) - np.arange(len(sorted_pvalues))) * sorted_pvalues
)

adjusted_pvalues = np.minimum(adjusted_pvalues, 1.0)

attribute_results['holm_p_value'] = np.nan
attribute_results.loc[sorted_indices, 'holm_p_value'] = adjusted_pvalues

# Sort by adjusted p-value
attribute_results = attribute_results.sort_values('holm_p_value')

display(attribute_results.round(4))

,attribute,count_with,count_without,mean_with,mean_without,mean_difference,p_value,holm_p_value
6,bar,21,64,61.2954,46.7144,14.5810,0.0000,0.0001
5,hard,15,70,40.5090,52.4184,-11.9094,0.0000,0.0002
0,fruity,38,47,44.1197,55.3271,-11.2074,0.0002,0.0011
2,peanutyalmondy,14,71,63.6971,47.6784,16.0188,0.0032,0.0160
4,crispedricewafer,7,78,66.1703,48.8940,17.2762,0.0041,0.0165
7,pluribus,44,41,46.8228,54.0664,-7.2436,0.0232,0.0697
3,nougat,7,78,60.0519,49.4431,10.6088,0.0927,0.1746
1,caramel,14,71,57.3469,48.9305,8.4164,0.0873,0.1746


### 8.1 Attribute Comparison Results

After applying the Holm correction, five attributes showed statistically significant differences in mean popularity at the 5% significance level:

- **Bar:** candies with this attribute had a mean win percentage 14.58 percentage points higher.
- **Hard:** hard candies had a mean win percentage 11.91 percentage points lower.
- **Fruity:** fruity candies had a mean win percentage 11.21 percentage points lower.
- **Peanutyalmondy:** candies with this attribute had a mean win percentage 16.02 percentage points higher.
- **Crispedricewafer:** candies with this attribute had a mean win percentage 17.28 percentage points higher.

The attributes `pluribus`, `nougat`, and `caramel` did not show statistically significant differences after the Holm correction.

Some groups have relatively few observations, particularly `crispedricewafer` and `nougat`, which each contain only seven candies. Their estimated differences should therefore be interpreted with caution.

These results describe unadjusted associations. Candy attributes may overlap, so the observed differences cannot be interpreted as independent effects or causal relationships.

### 8.2 Attribute Correlation Analysis

This section examines the linear associations between candy attributes and popularity (`winpercent`) using Pearson's correlation coefficient.

The analysis includes all binary attributes and allows their associations with popularity to be compared on a common scale.

Correlation coefficients indicate the direction and strength of linear associations, but they do not establish causation or account for interactions between attributes.

In [20]:
# Define binary attributes
binary_columns = [
    'chocolate',
    'fruity',
    'caramel',
    'peanutyalmondy',
    'nougat',
    'crispedricewafer',
    'hard',
    'bar',
    'pluribus'
]

# Calculate correlations with popularity
correlation_results = (
    df[binary_columns + ['winpercent']]
    .corr()['winpercent']
    .drop('winpercent')
    .sort_values(ascending=False)
)

display(correlation_results.to_frame(name='correlation').round(4))

,correlation
chocolate,0.6365
bar,0.4299
peanutyalmondy,0.4062
crispedricewafer,0.3247
caramel,0.2134
nougat,0.1994
pluribus,-0.2474
hard,-0.3104
fruity,-0.3809


### 8.2 Correlation Analysis Results

Chocolate showed the strongest positive correlation with popularity (r = 0.6365), followed by bar (r = 0.4299) and peanutyalmondy (r = 0.4062).

Fruity showed the strongest negative correlation (r = -0.3809), followed by hard (r = -0.3104).

These results are consistent with several patterns observed in the group comparisons performed in Section 8.1.

An interesting distinction can be observed for crispedricewafer: although it showed the largest difference in mean popularity between groups, its correlation with win percentage was lower than that of chocolate. This illustrates that correlation coefficients and raw mean differences measure different aspects of the data.

Since the attributes are binary, Pearson's correlation is equivalent to the point-biserial correlation in this context.

These correlations represent unadjusted associations. They do not account for overlapping attributes, interactions, or potential confounding factors, and should not be interpreted as evidence of causation.

## 9. Results and Discussion

This section brings together the findings from the statistical analyses to evaluate the research questions and discuss their implications.

The results are interpreted in terms of statistical significance, effect magnitude, and observed associations, while considering the limitations of the dataset and the methods used.

### 9.1 Chocolate and Popularity

The comparison between chocolate and non-chocolate candies revealed a statistically significant difference in mean popularity.

Chocolate candies had a mean win percentage of 60.92%, compared to 42.14% for non-chocolate candies, representing a difference of approximately 18.78 percentage points.

Welch's t-test indicated strong statistical evidence against the null hypothesis (t = 7.3031, p = 4.1635e-10). The Mann–Whitney U test also indicated a statistically significant difference between the groups (U = 1542, p = 6.9349e-09).

The 95% confidence interval for the difference in means ranged from 13.65 to 23.91 percentage points.

The estimated Hedges' g of 1.63 indicates a large standardized difference between the groups, although this estimate uses a pooled standard deviation as its standardizer.

Together, these results provide consistent evidence that chocolate candies had higher popularity scores in the analyzed dataset.

### 9.2 Other Candy Attributes

The additional attribute comparisons revealed statistically significant differences for five attributes after applying the Holm correction:

- **Bar:** positive difference of approximately 14.58 percentage points.
- **Peanutyalmondy:** positive difference of approximately 16.02 percentage points.
- **Crispedricewafer:** positive difference of approximately 17.28 percentage points.
- **Hard:** negative difference of approximately 11.91 percentage points.
- **Fruity:** negative difference of approximately 11.21 percentage points.

The correlation analysis also identified chocolate as the attribute most strongly associated with popularity (r = 0.6365).

Other positive associations included bar (r = 0.4299), peanutyalmondy (r = 0.4062), and crispedricewafer (r = 0.3247).

Fruity (r = -0.3809) and hard (r = -0.3104) showed negative associations with popularity.

Although these findings provide additional insight into candy characteristics, they represent unadjusted associations. The attributes may overlap, and their individual relationships with popularity cannot be interpreted as independent effects.

### 9.3 Overall Discussion

The statistical analyses support the patterns initially observed during exploratory data analysis.

Chocolate consistently showed the strongest positive association with popularity and a substantial difference in mean win percentage compared to non-chocolate candies.

Other attributes also presented meaningful differences, but their results should be interpreted with consideration for sample sizes, overlapping characteristics, and multiple comparisons.

The findings describe patterns within the 85 candy products included in the dataset. They do not establish causal relationships or demonstrate how these characteristics influence actual consumer purchasing behavior.

## 10. Conclusion

This project investigated the relationships between candy attributes and popularity using descriptive statistics, hypothesis testing, confidence intervals, effect size measurements, and correlation analysis.

The statistical analysis addressed the research questions as follows:

1. **Chocolate and popularity:** Chocolate candies had a statistically significantly higher mean win percentage than non-chocolate candies. The estimated difference was approximately 18.78 percentage points.

2. **Fruity and non-fruity candies:** Fruity candies had a lower mean win percentage than non-fruity candies, with a difference of approximately 11.21 percentage points. This difference remained statistically significant after applying the Holm correction.

3. **Magnitude of the differences:** The comparison between chocolate and non-chocolate candies showed a large standardized difference, with a Hedges' g of approximately 1.63. Other attributes also presented notable differences in mean popularity.

4. **Attributes most strongly associated with popularity:** Chocolate showed the strongest positive correlation with win percentage (r = 0.6365), while fruity showed the strongest negative correlation (r = -0.3809).

Overall, the findings indicate that candy characteristics are associated with differences in popularity within the analyzed dataset. Chocolate stood out as the attribute most strongly associated with higher popularity scores.

However, these results should be interpreted as statistical associations rather than causal relationships. Further research using larger datasets and multivariate methods could help investigate how different candy attributes interact and contribute to popularity.

## 11. Limitations

The findings of this statistical analysis should be interpreted in light of the following limitations:

- **Sample size:** The dataset contains only 85 candy products. Some attributes are represented by very small groups, which may reduce the precision of statistical estimates.

- **Measurement of popularity:** The `winpercent` variable represents the percentage of wins in pairwise candy preference comparisons. It does not directly measure sales, market share, or actual purchasing behavior.

- **Attribute overlap:** Candy characteristics are not mutually exclusive. A candy may contain multiple attributes, meaning that the observed associations do not represent isolated or independent effects.

- **Multiple comparisons:** Although the Holm correction was applied to the additional attribute comparisons, statistical testing across multiple characteristics still requires careful interpretation.

- **Methodological scope:** The analyses primarily compare groups independently and examine pairwise correlations. They do not account for interactions or adjust for other candy characteristics through multivariate models.

- **Generalizability:** The results describe the products included in this dataset and should not be assumed to represent all candies, consumers, or markets.

- **Correlation and causation:** Statistical associations do not establish that a particular candy attribute causes higher or lower popularity.

These limitations indicate that the results are useful for identifying patterns and generating hypotheses, but further research with larger and more diverse datasets would be necessary to support broader conclusions.